# Lab 10 solution, notebook 1: say no to a big order, and charge for delivery with a receipt

The worked answers to the two challenge tasks of `10a_meet_langgraph.ipynb`. The setup repeats the notebook's
nodes, routes and Part D's graph, so this file runs on its own. It calls no model and needs no key.

## Setup

In [1]:
import operator
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

BASE_PRICE = {"small": 8.00, "medium": 11.00, "large": 14.00}
EXTRA_PRICE = 1.50
ASK_ABOVE = 18.00


class Order(TypedDict):
    size: str
    extras: list
    toppings: Annotated[list, operator.add]
    price: float
    approved: bool
    status: str

In [2]:
def take_order(order):
    return {"toppings": ["tomato sauce", "mozzarella"]}


def add_extras(order):
    return {"toppings": order["extras"]}


def price(order):
    return {"price": BASE_PRICE[order["size"]] + EXTRA_PRICE * len(order["extras"])}


def ask_first(order):
    answer = interrupt(f"This order comes to ${order['price']:.2f}. Go ahead?")
    return {"approved": answer}


def place_order(order):
    print(f"🍕 ORDER PLACED: {order['size']} with {order['toppings']}, ${order['price']:.2f}")
    return {"status": "in the oven"}


def route(order):
    if order["price"] > ASK_ABOVE:
        return "ask_first"
    return "place_order"


def after_asking(order):
    if order["approved"]:
        return "place_order"
    return END

Part D's graph, with a checkpointer:

In [3]:
builder = StateGraph(Order)

builder.add_node("take_order", take_order)
builder.add_node("add_extras", add_extras)
builder.add_node("price", price)
builder.add_node("ask_first", ask_first)
builder.add_node("place_order", place_order)

builder.add_edge(START, "take_order")
builder.add_edge("take_order", "add_extras")
builder.add_edge("add_extras", "price")
builder.add_conditional_edges("price", route, ["ask_first", "place_order"])
builder.add_conditional_edges("ask_first", after_asking, ["place_order", END])
builder.add_edge("place_order", END)

graph = builder.compile(checkpointer=InMemorySaver())

print(list(graph.get_graph().nodes))

['__start__', 'take_order', 'add_extras', 'price', 'ask_first', 'place_order', '__end__']


## Task 1: Say no

The party order costs 20.00 dollars, so it stops at `ask_first`. We send it under a new thread id, `order-19`.

In [4]:
PARTY = {"size": "large", "extras": ["mushrooms", "olives", "peppers", "onions"]}

config = {"configurable": {"thread_id": "order-19"}}

paused = graph.invoke(PARTY, config)

print("question:", paused["__interrupt__"][0].value)

question: This order comes to $20.00. Go ahead?


The customer says no. The answer travels the same way as a yes: `Command(resume=...)`, with the same `config`.

In [5]:
done = graph.invoke(Command(resume=False), config)

print("approved:    ", done["approved"])
print("has a status:", "status" in done)
print("next node:   ", graph.get_state(config).next)

approved:     False
has a status: False
next node:    ()


No `ORDER PLACED` line was printed. On the resume, `interrupt(...)` returned `False`, `ask_first` saved it under
`approved`, and `after_asking` returned `END`. `place_order` never ran, so nothing wrote a `status`: the key is
missing from the result. `next` is empty, so the thread is finished. It ended without an order.

The same two lines of code carried a yes in the notebook and a no here. The graph does not treat an answer as an
approval. Our route reads the answer and decides.

## Task 2: Charge for delivery, and keep a receipt

Three changes:

1. The state gets a key `receipt`, a list with the reducer `operator.add`, so that every node can add a line.
2. `price` returns a receipt line next to the price.
3. A new node, `add_delivery`, adds 2.50 dollars to the price and returns a second receipt line.

`price` and `add_delivery` both return `price` and both return `receipt`. The two keys are merged differently:
`price` has no reducer, so the newer value replaces the older one, and `receipt` has one, so the lines add up.

In [6]:
DELIVERY = 2.50


class Order(TypedDict):
    size: str
    extras: list
    toppings: Annotated[list, operator.add]
    price: float
    receipt: Annotated[list, operator.add]      # new
    approved: bool
    status: str


def price(order):
    amount = BASE_PRICE[order["size"]] + EXTRA_PRICE * len(order["extras"])
    return {"price": amount, "receipt": [f"pizza {amount:.2f}"]}


def add_delivery(order):
    return {"price": order["price"] + DELIVERY, "receipt": [f"delivery {DELIVERY:.2f}"]}

In the wiring, `add_delivery` comes after `price`, and the conditional edge moves: `route` now runs after
`add_delivery`, so it reads the price with delivery.

In [7]:
builder = StateGraph(Order)

builder.add_node("take_order", take_order)
builder.add_node("add_extras", add_extras)
builder.add_node("price", price)
builder.add_node("add_delivery", add_delivery)                                      # new
builder.add_node("ask_first", ask_first)
builder.add_node("place_order", place_order)

builder.add_edge(START, "take_order")
builder.add_edge("take_order", "add_extras")
builder.add_edge("add_extras", "price")
builder.add_edge("price", "add_delivery")                                           # new
builder.add_conditional_edges("add_delivery", route, ["ask_first", "place_order"])  # moved
builder.add_conditional_edges("ask_first", after_asking, ["place_order", END])
builder.add_edge("place_order", END)

graph = builder.compile(checkpointer=InMemorySaver())

A medium with mushrooms and olives:

In [8]:
medium = graph.invoke({"size": "medium", "extras": ["mushrooms", "olives"]},
                      {"configurable": {"thread_id": "order-20"}})

print("price:  ", medium["price"])
print("receipt:", medium["receipt"])

🍕 ORDER PLACED: medium with ['tomato sauce', 'mozzarella', 'mushrooms', 'olives'], $16.50
price:   16.5
receipt: ['pizza 14.00', 'delivery 2.50']


The order is placed at 16.50 dollars: 14.00 for the pizza and 2.50 for delivery, the two lines of the receipt.
16.50 is under the limit of 18 dollars, so `route` sent it straight to the kitchen.

Now a large with the same extras:

In [9]:
config = {"configurable": {"thread_id": "order-21"}}

large = graph.invoke({"size": "large", "extras": ["mushrooms", "olives"]}, config)

print("receipt: ", large["receipt"])
print("question:", large["__interrupt__"][0].value)

receipt:  ['pizza 17.00', 'delivery 2.50']
question: This order comes to $19.50. Go ahead?


No `ORDER PLACED` line. The pizza alone costs 17.00 dollars, which is under the limit. With delivery it comes to
19.50, and `route` now runs after `add_delivery`, so the order stops at `ask_first`. Where an edge sits in the
graph decides which price the rule is applied to.

The customer says yes:

In [10]:
done = graph.invoke(Command(resume=True), config)

print("price:", done["price"], "| status:", done["status"])

🍕 ORDER PLACED: large with ['tomato sauce', 'mozzarella', 'mushrooms', 'olives'], $19.50
price: 19.5 | status: in the oven


The order is placed at 19.50 dollars.